In [ ]:
"""
Author: Sophie A. Liu
Date: 05/11/2026
Purpose: basic odds-ratios modeling for visualization
"""

In [ ]:
# necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [22]:
# working directory
os.chdir("i:/Hu Lab/Sophie/1. Cell death/visium image manual spot selection/20260413_final_merge/data")

# my dataset & then setting predictors and response variable
df = pd.read_csv("0511iso_sig_means.csv")
# df = pd.read_csv("0511pd_sig_means.csv")

gene_cols = df.columns[11:61]
X = df[gene_cols]
y = df["prop_dying"]

In [23]:
model = Pipeline([
    ("scaler", StandardScaler()),            # transform each feature to have same scale
    ("enet", ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],  # mixing parameter, smaller is more lasso (less important = 0), 
                                             # larger is more ridge (shrinking less important and keeping ALL features)
        alphas=None,
        cv=5,              # trains on 4 folds, tests on 1 fold & repeat for all 
        max_iter=10000,
        n_jobs=-1
    ))
])

In [ ]:
model.fit(X, y)

In [ ]:
enet = model.named_steps["enet"]

coeffs = enet.coef_
intercept = enet.intercept_

for name, coef in zip(gene_cols, coeffs):
    print(f"{name}: {coef:.4f}")

In [ ]:
# checking characteristics
enet.l1_ratio_ # returns % lasso. if it's smaller --> more correlated genes togetther
# enet.alpha_

In [ ]:
# barcharts
mask = abs(coeffs) > 0.05

coeffs_nz = coeffs[mask]
gene_cols_nz = gene_cols[mask]

iso7_df = pd.Series(coeffs_nz, index=gene_cols_nz) # obtained from running it again & changing dataset.
                                                   # renamed vars to maintain previous
# pd1_df = pd.Series(coeffs_nz, index=gene_cols_nz)                                        

# Plotting
plt.figure(figsize=(8, 5))
plt.barh(gene_cols_nz, iso7_df, alpha=0.2, label='isotype control', color='blue')
# plt.barh(gene_cols_nz, pd1_df, alpha=0.2, label='anti-PD1-treated', color='red')

# Customization
plt.legend()
plt.xlabel("standardized scale")
plt.xlim(-0.6, 0.6)
plt.title("Elastic net coefficients")
#plt.tick_params(axis='y', labelsize=5)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import cross_validate

# what is this model's capability to distinguish these programs as predictors of dying areas?
scores = cross_validate(
    model, X, y,
    cv=5,
    scoring=["r2", "neg_mean_squared_error"]
)

print(scores["test_r2"].mean())
print(scores["test_neg_mean_squared_error"].mean())

print(scores["test_r2"].mean())
print(scores["test_roc_auc"].mean())